<img src="../static/imo_health.png" alt="IMO Health Logo" width="300"/>

---

# Entity Context Identification Workflow

This notebook demonstrates strategies for identifying and mapping context to clinical entities extracted from medical notes.

## Overview
The goal is to determine the most relevant context for each entity based on the section it appears in and related information from other sections of the clinical note.

In [ ]:
#pip install -r requirements.txt

## Step 1: Load Full Clinical Note

This step loads the complete clinical note text file that will be analyzed. The note contains structured medical documentation including sections like History of Present Illness (HPI), Assessment, Plan, etc.

**Purpose:** Provides the full source text from which entities will be extracted and contextualized.

**Key Actions:**
- Loads the clinical note from a text file (e.g., `sample.txt`)
- Loads expected ICD-10 diagnosis codes from a gold standard JSON file for accuracy evaluation
- Displays note statistics (length, preview) for validation

In [ ]:
import pandas as pd
import re
from typing import List, Dict, Tuple
import os
import json

# Load clinical note from file
# Update the file path below to point to your clinical note text file
note_id = "sample2"  # Example note ID, change as needed
note_file_path = f"{note_id}.txt"  # Change this to your file path

try:
    with open(f"{note_id}_gold.json", 'r', encoding='utf-8') as f:
        expected_icd10_codes = json.load(f)
except:
    print("No gold standard file found. Proceeding without expected ICD0-10 codes. ")
    expected_icd10_codes = []
if not os.path.exists(note_file_path):
    print(f"Error: File '{note_file_path}' not found.")
    print("Please update the note_file_path variable to point to your clinical note file.")
    clinical_note = ""
else:
    with open(note_file_path, 'r', encoding='utf-8') as f:
        clinical_note = f.read()
    
    print("Clinical note loaded successfully")
    print(f"File: {note_file_path}")
    print(f"Note length: {len(clinical_note)} characters")
    print(f"\nFirst 200 characters:")
    print(clinical_note[:200] + "...")

## Step 2: Load Problem Entities via CSV

This step loads pre-extracted clinical entities (problems, diagnoses, symptoms) from a CSV file. These entities have been identified using a Named Entity Recognition (NER) system or entity extraction tool.

**Purpose:** Provides the list of medical concepts detected in the clinical note that need to be normalized to standardized codes.

**Key Data:**
- **entity**: The text span of the clinical term (e.g., "chest pain", "hypertension")
- **offset**: Character position where the entity starts in the note
- **length**: Number of characters in the entity text
- **Semantic Tag**: Entity type (e.g., problem, medication, procedure)

**Note:** The CSV file is generated using IMO's entity extraction API or similar NER tools.

In [ ]:
# Load entities from CSV file
# Update the file path below to point to your entities CSV file
entities_csv_path = f"{note_id}_entities.csv"  # Change this to your CSV file path

if not os.path.exists(entities_csv_path):
    print(f"Error: File '{entities_csv_path}' not found.")
    print("Please update the entities_csv_path variable to point to your CSV file.")
    print("You can generate this file using: python extract_entities.py input1.txt output.csv")
    entities_df = pd.DataFrame(columns=['entity', 'offset', 'length'])
else:
    entities_df = pd.read_csv(entities_csv_path)
    
    print("Entities loaded successfully")
    print(f"File: {entities_csv_path}")
    print(f"Total entities: {len(entities_df)}")
    print(f"\nFirst 10 entities:")
    print(entities_df.head(10))
    
    # Display entity type distribution if available
    if 'entity' in entities_df.columns:
        print(f"\nDataFrame columns: {list(entities_df.columns)}")
        print(f"Data types:\n{entities_df.dtypes}")

## Step 3: Find Section Headers for Each Entity

This step identifies the clinical note sections (e.g., HPI, Assessment, Plan) and maps each extracted entity to its parent section.

**Why This Matters:** Section context helps determine which entities should receive additional contextual enrichment during normalization.

**Purpose:** Section context is critical for understanding entity relevance. For example:

- Entities in **Assessment** or **Impression** sections typically represent primary diagnoses for the current encounter
- Entities in **Past Medical History** represent chronic or historical conditions
- Entities in **Family History** represent hereditary conditions not affecting the patient

**Process:**

1. Scans the clinical note for standard SOAP note section headers using regex patterns
2. Creates a mapping of section names to their character positions in the text
3. Assigns each entity to its containing section based on the entity's offset position
4. Generates a CSV file with entities and their mapped sections

This section mapping enables the context-building strategies in Step 4 to make intelligent decisions about which entities benefit from contextual enrichment.

In [ ]:
def identify_sections(text: str) -> Dict[str, Tuple[int, int]]:
    """
    Identify section headers and their positions in the clinical note.
    
    Returns:
        Dictionary mapping section names to (start_pos, end_pos) tuples
    """
    # Common section headers in clinical notes
    section_patterns = [
        r"^ASSESSMENT[\:\n]",
        r"^ASSESSMENT AND PLAN[\:\n]",
        r"^CHIEF COMPLAINT[\:\n]",
        r"^CURRENT MEDICATIONS[\:\n]",
        r"^FAMILY HISTORY[\:\n]",
        r"^FOLLOW-UP[\:\n]",
        r"^HISTORY[\:\n]",
        r"^HISTORY OF PRESENT ILLNESS[\:\n]",
        r"^HISTORY OF PRESENT ILLNESS \(HPI\)[\:\n]",
        r"^HPI[\:\n]",
        r"^IMPRESSION[\:\n]",
        r"^MEDICAL DECISION MAKING[\:\n]",
        r"^OBJECTIVE[\:\n]",
        r"^PAST MEDICAL HISTORY[\:\n]",
        r"^PATIENT EDUCATION[\:\n]",
        r"^PATIENT INSTRUCTIONS[\:\n]",
        r"^PLAN[\:\n]",
        r"^PHYSICAL EXAM[\:\n]",
        r"^PHYSICAL EXAMINATIONS[\:\n]",
        r"^PRESENTING COMPLAINT[\:\n]",
        r"^PROBLEM LIST[\:\n]",
        r"^REASON[\:\n]",
        r"^REASON FOR VISIT[\:\n]",
        r"^REVIEW OF SYSTEMS[\:\n]",
        r"^REVIEW OF SYSTEMS \(ROS\)[\:\n]",
        r"^SOCIAL HISTORY[\:\n]",
        r"^SUBJECTIVE[\:\n]",
        r"^SURGICAL HISTORY[\:\n]",       
        
    ]
    
    sections = {}
    positions = []
    
    # Find all section headers
    for pattern in section_patterns:
        match = re.search(pattern, text, re.IGNORECASE | re.MULTILINE)
        if match:
            print(f"Found section header: '{match.group().strip()}' at position {match.start()}")
            section_name = match.group().strip(':').strip()
            positions.append((match.start(), section_name))
    
    # Sort by position
    positions.sort()
    
    # Calculate section ranges
    for i, (start_pos, section_name) in enumerate(positions):
        if i < len(positions) - 1:
            end_pos = positions[i + 1][0]
        else:
            end_pos = len(text)
        sections[section_name] = (start_pos, end_pos)
    
    return sections

def find_entity_section(entity_offset: int, sections: Dict[str, Tuple[int, int]]) -> str:
    """
    Find which section an entity belongs to based on its offset.
    """
    for section_name, (start, end) in sections.items():
        if start <= entity_offset < end:
            return section_name
    return "Unknown"

# Identify sections
sections = identify_sections(clinical_note)
print("Identified sections:")
for section, (start, end) in sections.items():
    print(f"  {section}: characters {start}-{end}")

# Map entities to sections
entities_df['section'] = entities_df['offset'].apply(
    lambda x: find_entity_section(x, sections)
)

print("\nEntities with sections:")
print(entities_df[['entity', 'section']])

# Save entities with sections to CSV
output_with_sections = "entities_with_sections.csv"
entities_df.to_csv(output_with_sections, index=False)
print(f"\n✓ Entities with sections saved to: {output_with_sections}")
print(f"Total rows saved: {len(entities_df)}")

# Show section distribution
print("\nEntity distribution by section:")
section_counts = entities_df['section'].value_counts()
for section, count in section_counts.items():
    print(f"  {section}: {count} entities")

## Step 4: Strategies for Mapping Entity to Context

This section defines helper functions for entity normalization using IMO Precision Normalize API. The normalization process converts free-text clinical terms into standardized medical codes (ICD-10, SNOMED CT, etc.).

### Context Strategy Definition

Different clinical note sections have varying diagnostic significance.

**HIGH-VALUE SECTIONS (Clinical SME Analysis)**
- **Assessment**
- **Assessment & Plan**
- **Problem List**
- **HPI (History of Present Illness)**
- **History of Present Illness**
- **Physical Exam**
- **History**
- **Review of Systems**
- **Reason for Visit**
- **Reason**
- **Medical Decision Making**
- **Patient Instructions**
- **Follow-up**
- **Presenting Complaint**

These sections are prioritized for contextual enrichment during normalization.

**OTHER/HISTORICAL SECTIONS** (typically normalized without additional enrichment unless needed):
- **Past Medical History**
- **Family/Social/Surgical History**
- **Current Medications**
- Other non-diagnostic or administrative sections

### Rationale

Contextual enrichment improves normalization accuracy for diagnostically significant entities while avoiding over-specification of historical or non-relevant conditions.

### Core Functions

**`fetch_token()`**: Authenticates with IMO API using OAuth2 client credentials

**`normalize()`**: 
- Routes entities to appropriate normalization endpoints
- Entities WITH context → Enrichment endpoint (batch size: 5, returns top 3 matches)
- Entities WITHOUT context → Base endpoint (batch size: 20, returns top 1 match)
- Returns aggregated normalization results with matched terms and codes

**`summarize_results()`**: 
- Displays normalization results in an HTML table format
- Shows input term, context used, matched terms, ICD-10 codes, and confidence scores

**`calculate_accuracy()`**: 
- Calculates accuracy metrics when gold standard codes are available
- Compares normalized ICD-10 codes against expected gold standard codes
- Reports true positives, false positives, false negatives
- Calculates accuracy percentage

In [ ]:
import json, requests, uuid

normalize_url = "https://api.imohealth.com/"
client_id = ""
client_secret = ""
HIGH_VALUE_SECTIONS = set([
    "assessment",
    "assessment and plan",
    "problem list",
    "medical decision making",
    "history of present illness",
    "hpi",
    "history",
    "physical exam",
    "review of systems",
    "reason",
    "reason for visit",
    "follow-up",
    "presenting complaint",
    "patient instructions"
    ])

def normalize_section_name(name: str) -> str:
        """Normalize section names to handle variations"""
        name_lower = name.lower().strip()
        if "assessment" in name_lower and "plan" in name_lower:
            return "assessment and plan"
        elif "assessment" in name_lower:
            return "assessment"
        elif name_lower in ["hpi", "history of present illness", "history of present illness (hpi)"]:
            return "hpi"
        elif "problem list" in name_lower:
            return "problem list"
        elif "physical exam" in name_lower:
            return "physical exam"
        elif name_lower == "history":
            return "history"
        elif "reason for visit" in name_lower or name_lower == "reason":
            return "reason for visit"
        elif "review of systems" in name_lower:
            return "review of systems"
        elif "medical decision making" in name_lower:
            return "medical decision making"
        return name_lower
    
SECTION_PRIORITIES = {
        "assessment": ["assessment", "assessment and plan", "hpi", "problem list", "history", "physical exam"],
        "assessment and plan": ["assessment", "assessment and plan", "hpi", "problem list", "history", "physical exam"],
        "problem list": ["problem list", "assessment", "assessment and plan", "hpi", "reason for visit"],
        "hpi": ["hpi", "assessment", "assessment and plan", "problem list", "physical exam", "review of systems", "medical decision making"],
        "history of present illness": ["hpi", "assessment", "assessment and plan", "problem list", "physical exam", "review of systems", "medical decision making"],
        "history of present illness (hpi)": ["hpi", "assessment", "assessment and plan", "problem list", "physical exam", "review of systems", "medical decision making"],
        "physical exam": ["physical exam", "problem list", "hpi"],
        "physical examinations": ["physical exam", "problem list", "hpi"],
        "history": ["history", "problem list"],
        "reason": ["reason for visit", "assessment", "assessment and plan", "hpi"],
        "reason for visit": ["reason for visit", "assessment", "assessment and plan", "hpi"],
        "review of systems": ["review of systems", "hpi", "assessment", "assessment and plan"],
        "chief complaint": ["chief complaint", "hpi", "assessment", "assessment and plan"],
        "impression": ["impression", "hpi", "problem list"],
        "medical decision making": ["medical decision making", "assessment", "assessment and plan", "hpi"],
        "presenting complaint": ["presenting complaint", "hpi", "assessment", "assessment and plan"],
        "patient instructions": ["patient instructions", "assessment", "assessment and plan", "plan"],
    }
    
DEFAULT_PRIORITIES = [
        "assessment", "assessment and plan", "hpi", "problem list", 
        "physical exam", "plan", "past medical history", "history"
    ]
    

def fetch_token(client_id: str, client_secret: str) -> str:
    url = f"{normalize_url}oauth/token"
    payload = {
        'grant_type': 'client_credentials',
        'client_id': client_id,
        'client_secret': client_secret
    }
    response = requests.post(url, data=payload)
    response.raise_for_status()
    token = response.json().get('access_token')
    return token

def normalize(entity_context_pairs: List[Dict[str, str]]):
    """
    Normalize entities, automatically routing to the appropriate endpoint based on context.
    
    Entities with context are sent to the enrichment endpoint (current batch size 1 per API call).
    Entities without context are sent to the base normalize endpoint (batch size 20).
    
    Args:
        entity_context_pairs: List of dicts with 'entity' and 'context' keys.
                             Context can be empty string for base normalization.
    
    Returns:
        Combined API response with all results aggregated
    """
    token = fetch_token(client_id, client_secret)
    headers = {
        'Authorization': f'Bearer {token}',
        'Content-Type': 'application/json',
    }
    
    # Separate entities by whether they have context
    with_context = [pair for pair in entity_context_pairs if pair.get("context", "").strip()]
    without_context = [pair for pair in entity_context_pairs if not pair.get("context", "").strip()]
    
    all_results = []
    total_count = 0
    total_error_count = 0
    record_id_counter = 10000
    
    # Process entities WITH context (enrichment endpoint, current batch size 5)
    if with_context:
        ENRICHMENT_BATCH_SIZE = 5
        enrichment_url = f"{normalize_url}precision/normalize/enrichment"
        
        for batch_start in range(0, len(with_context), ENRICHMENT_BATCH_SIZE):
            batch = with_context[batch_start:batch_start + ENRICHMENT_BATCH_SIZE]
            
            request = {
                "client_request_id": str(uuid.uuid4()),
                "preferences": {
                    "threshold": 0.0, 
                    "match_field_pref": "input_term",
                    "debug" : True,
                    "size" : 3
                },
                "requests": [
                    {
                        "record_id": str(record_id_counter + i),
                        "domain": "Problem",
                        "input_term": pair["entity"],
                        "context": {
                            "source_text": pair["context"]
                        }
                    }
                    for i, pair in enumerate(batch)
                ]
            }
            response = requests.post(enrichment_url, headers=headers, json=request)
            response.raise_for_status()
            batch_result = response.json()
            all_results.extend(batch_result.get("requests", []))
            total_count += batch_result.get("summary", {}).get("count", 0)
            total_error_count += batch_result.get("summary", {}).get("error_count", 0)
            record_id_counter += len(batch)
    
    # Process entities WITHOUT context (base normalize endpoint, batch size 20)
    if without_context:
        BASE_BATCH_SIZE = 20
        base_url = f"{normalize_url}precision/normalize"
        
        for batch_start in range(0, len(without_context), BASE_BATCH_SIZE):
            batch = without_context[batch_start:batch_start + BASE_BATCH_SIZE]
            
            request = {
                "client_request_id": str(uuid.uuid4()),
                "preferences": {
                    "size": 1,
                    "threshold": 0
                },
                "requests": [
                    {
                        "record_id": str(record_id_counter + i),
                        "domain": "Problem",
                        "input_term": pair["entity"]
                    }
                    for i, pair in enumerate(batch)
                ]
            }
            response = requests.post(base_url, headers=headers, json=request)
            response.raise_for_status()
            batch_result = response.json()
            all_results.extend(batch_result.get("requests", []))
            total_count += batch_result.get("summary", {}).get("count", 0)
            total_error_count += batch_result.get("summary", {}).get("error_count", 0)
            record_id_counter += len(batch)
    
    # Return combined result
    return {
        "summary": {
            "count": total_count,
            "error_count": total_error_count
        },
        "requests": all_results
    }

def summarize_results(results: Dict[str,Dict]):
    """
    Summarize and display normalization results in an HTML table.
    Displays results with proper formatting and newline handling.
    Also calculates and displays accuracy metrics.
    
    Returns:
        Dictionary with accuracy metrics from calculate_accuracy()
    """
    from IPython.display import display, HTML
    
    requests = results.get("requests", [])
    
    # Extract all ICD-10 codes from the results
    found_icd10_codes = []
    for request in requests:
        top_matches = request.get("response", {}).get("items", [])
        for match in top_matches:
            icd10_codes = match.get("metadata", {}).get("mappings", {}).get("icd10cm", {}).get("codes", [])
            for code_obj in icd10_codes:
                found_icd10_codes.append(code_obj["code"])
    
    # Create HTML table with left-aligned text and proper newline rendering
    html = "<style>table.results td {text-align: left; vertical-align: top; white-space: pre-wrap;}</style>"
    html += '<table class="results" border="1" style="border-collapse: collapse; width: 100%;">'
    html += '<thead><tr>'
    html += '<th style="text-align: left; padding: 8px; background-color: rgba(128, 128, 128, 0.2);">Input Term</th>'
    html += '<th style="text-align: left; padding: 8px; background-color: rgba(128, 128, 128, 0.2);">Context</th>'
    html += '<th style="text-align: left; padding: 8px; background-color: rgba(128, 128, 128, 0.2);">Matches</th>'
    html += '</tr></thead><tbody>'
    
    for request in requests:
        input_term = request.get("input_term", "")
        context_text = request.get("context", {}).get("source_text", "")
        top_matches = request.get("response", {}).get("items", [])
        
        html += '<tr>'
        html += f'<td style="padding: 8px; border: 1px solid #ddd;">{input_term}</td>'
        html += f'<td style="padding: 8px; border: 1px solid #ddd;">{context_text if context_text else "None"}</td>'
        
        # Format matches
        if top_matches:
            matches_html = ""
            for rank, match in enumerate(top_matches, 1):
                default_title = match.get("default_lexical_title", "N/A")
                icd10_codes = match.get("metadata", {}).get("mappings", {}).get("icd10cm", {}).get("codes", [])
                icd10_codes_str = ", ".join([code["code"] for code in icd10_codes]) if icd10_codes else "None"
                score = match.get("score", "N/A")
                score_str = f"{score:.2f}" if isinstance(score, (int, float)) else score
                
                matches_html += f"{rank}. {default_title}\n   ICD-10: {icd10_codes_str}\n   Score: {score_str}"
                if rank < len(top_matches):
                    matches_html += "\n\n"
            html += f'<td style="padding: 8px; border: 1px solid #ddd;">{matches_html}</td>'
        else:
            explanation = request.get("response", {}).get("explanation", "No match")
            html += f'<td style="padding: 8px; border: 1px solid #ddd;">No match found\n({explanation})</td>'
        
        html += '</tr>'
    
    html += '</tbody></table>'
    
    print("\n=== Normalization Results ===")
    display(HTML(html))
    
    # Calculate and display accuracy if expected codes are available
    accuracy_results = None
    if 'expected_icd10_codes' in globals() and expected_icd10_codes:
        accuracy_results = calculate_accuracy(expected_icd10_codes, found_icd10_codes)
        print(json.dumps(accuracy_results, indent=2))
    
    return accuracy_results

def calculate_accuracy(expected_icd10_codes: List[str], found_icd10_codes: List[str]):
    """
    Print a report detailing the accuracy of the normalization results.
    
    Args:
        expected_icd10_codes: List of ICD-10 codes that should be found
        found_icd10_codes: List of ICD-10 codes that were actually found
    """
    # Convert to sets for easier comparison
    expected_set = set(expected_icd10_codes)
    found_set = set(found_icd10_codes)
    
    # Calculate metrics
    true_positives = expected_set & found_set  # Found and expected
    false_negatives = expected_set - found_set  # Expected but not found
    false_positives = found_set - expected_set  # Found but not expected
    
    # Calculate percentage
    if len(expected_set) > 0:
        accuracy_percentage = (len(true_positives) / len(expected_set)) * 100
    else:
        accuracy_percentage = 0.0
    
    # Print report
    print("\n" + "="*80)
    print("NORMALIZATION ACCURACY REPORT")
    print("="*80)
    
    print(f"\nTotal Expected ICD-10 Codes: {len(expected_set)}")
    print(f"Total Found ICD-10 Codes: {len(found_set)}")
    print(f"Correctly Identified (True Positives): {len(true_positives)}")
    print(f"\nAccuracy: {accuracy_percentage:.2f}% ({len(true_positives)}/{len(expected_set)})")
    
    print("\n" + "-"*80)
    print("CODES CORRECTLY IDENTIFIED:")
    print("-"*80)
    if true_positives:
        for code in sorted(true_positives):
            print(f"  ✓ {code}")
    else:
        print("  None")
    
    print("\n" + "-"*80)
    print(f"CODES NOT FOUND (False Negatives): {len(false_negatives)}")
    print("-"*80)
    if false_negatives:
        for code in sorted(false_negatives):
            print(f"  ✗ {code}")
    else:
        print("  None")
    
    print("\n" + "-"*80)
    print(f"UNEXPECTED CODES FOUND (False Positives): {len(false_positives)}")
    print("-"*80)
    if false_positives:
        for code in sorted(false_positives):
            print(f"  ⚠ {code}")
    else:
        print("  None")
    
    print("\n" + "="*80)
    
    # Return metrics for potential further use
    return {
        "accuracy_percentage": accuracy_percentage,
        "true_positives": len(true_positives),
        "false_negatives": len(false_negatives),
        "false_positives": len(false_positives),
        "expected_total": len(expected_set),
        "found_total": len(found_set),
        "codes_correctly_identified": sorted(list(true_positives)),
        "codes_not_found": sorted(list(false_negatives)),
        "additional_codes_returned": sorted(list(false_positives))
    }


### Approach 0: Baseline - No Context

**Purpose:**  
Establishes a performance baseline to quantify the improvement gained from contextual enrichment strategies. All other approaches are compared against this baseline.

**Methodology:**
- Every entity is normalized WITHOUT any surrounding context
- Uses IMO Precision Normalize API base endpoint only
- No section information, sentence context, or related clinical details are included
- Returns single best match per entity based solely on string matching and semantic similarity

**Advantages:**
- Simple and fast
- No preprocessing required
- Consistent results

**Limitations:**
- Cannot disambiguate polysemous terms
- May miss important clinical nuance
- Lower specificity for complex medical conditions

**Example:**
- Input: "chest pain"
- Context: None
- Result: May return generic "Chest pain" code without differentiating between cardiac, musculoskeletal, or other etiologies

**Use Case:** Baseline comparison to measure the value-add of subsequent context-aware approaches.

In [ ]:
# Approach 0: Baseline - No context for any entity
entity_context_pairs = []
for entity, section in entities_df[['entity', 'section']].itertuples(index=False):
    print(f"Normalizing entity '{entity}' in section '{section}' - NO CONTEXT")
    entity_context_pairs.append({
        "entity": entity,
        "context": ""  # Empty context for all entities
    })

print(f"\nTotal entities to normalize: {len(entity_context_pairs)}")
print("All entities will be sent without context (baseline approach)")

approach_0_results = normalize(entity_context_pairs)
approach_0_accuracy = summarize_results(approach_0_results)


### Approach 1: Dynamic Rules-Based Context with Section-Specific Priorities

**Overview:**  
An enhanced rule-based approach that dynamically selects the most valuable context sections based on research-driven insights about section-specific information needs. Unlike Approach 1's fixed prioritization, this approach adapts to where the entity is found.

**Methodology:**

1. **Research-Driven Section Mapping:**
   - Uses empirical data showing which context sections provide the most value for entities in each clinical section
   - Dynamic priority lists tailored to each entity location (Assessment, HPI, Problem List, etc.)

2. **Section-Specific Priority Rules:**
   - **Assessment & Plan entities**: Self-context → HPI → Problem List → History → Physical Exam
   - **Problem List entities**: Self-context → Assessment & Plan → HPI → Reason for Visit
   - **HPI entities**: Self-context → Assessment & Plan → Problem List → Physical Exam → Review of Systems → Medical Decision Making
   - **Physical Exam entities**: Self-context → Problem List → HPI
   - **History entities**: Self-context → Problem List
   - **Other sections**: Fallback to Approach 1's default prioritization

3. **Hierarchical Context Building:**
   - Starts with entity's immediate section (self-context proven most valuable)
   - Adds additional sections following research-based priority order
   - Builds up to 1,000-character limit

4. **Intelligent Section Matching:**
   - Normalizes section names (case-insensitive, handles variations like "HPI" vs "History of Present Illness")
   - Gracefully handles missing sections

**Advantages:**
- ✅ Research-backed prioritization improves relevance
- ✅ Self-context emphasis leverages section-specific information density
- ✅ Adaptive to entity location (HPI entities get A&P context, A&P entities get HPI context)
- ✅ Fast and deterministic like Approach 1
- ✅ No ML infrastructure required

**Limitations:**
- ❌ Priority rules derived from specific dataset may not generalize to all clinical settings
- ❌ Still includes entire sections (may contain some irrelevant content)
- ❌ Requires section headers to be correctly identified
- ❌ Fixed rules per section (not adaptive to individual entity characteristics)



In [ ]:
def build_context_approach_1(section: str, full_text: str, sections: Dict[str, Tuple[int, int]]) -> str:
    """
    Build context using dynamic section prioritization based on entity's section location.
    Uses research-driven insights about which context sections are most valuable for each entity section.
    """
    if section.lower().strip() not in HIGH_VALUE_SECTIONS:
        return ""
    
    normalized_entity_section = normalize_section_name(section)
    sections_by_priority = SECTION_PRIORITIES.get(normalized_entity_section, DEFAULT_PRIORITIES)
    
    if section in sections:
        section_start = sections[section][0]
        section_end = sections[section][1]
        context = full_text[section_start:section_end].strip()
    else:
        context = ""
    
    available_sections = {}
    for section_name, section_range in sections.items():
        normalized_name = normalize_section_name(section_name)
        available_sections[normalized_name] = (section_name, section_range)
    
    print(f"Building context for entity in section '{section}'")
    print(f"Priority order: {sections_by_priority}")
    print(f"Available sections in note: {list(available_sections.keys())}")
    
    MAX_CONTEXT_LENGTH = 1000
    sections_added = [section]
    
    for priority_section in sections_by_priority:
        if priority_section == normalized_entity_section:
            continue
        if priority_section in available_sections:
            actual_section_name, (start, end) = available_sections[priority_section]
            if actual_section_name in sections_added:
                continue
            section_text = full_text[start:end].strip()
            if len(context) + len(section_text) + 2 <= MAX_CONTEXT_LENGTH:
                context += "\n\n" + section_text
                sections_added.append(actual_section_name)
                print(f"  Added section: {actual_section_name} ({len(section_text)} chars)")
            else:
                remaining = MAX_CONTEXT_LENGTH - len(context) - 2
                if remaining > 100:
                    context += "\n\n" + section_text[:remaining]
                    sections_added.append(actual_section_name)
                    print(f"  Added partial section: {actual_section_name} ({remaining} chars)")
                break
    
    print(f"Final context length: {len(context)} characters")
    print(f"Sections included: {sections_added}")
    return context

# Generate contexts for each entity using Approach 1
entity_context_pairs = []
for entity, section in entities_df[['entity', 'section']].itertuples(index=False):
    print(f"\nNormalizing entity '{entity}' in section '{section}'")
    entity_context = build_context_approach_1(section, clinical_note, sections)
    entity_context_pairs.append({
        "entity": entity,
        "context": entity_context
    })
    if not entity_context:
        print(f"No context built for '{entity}' as it is not in a high-value section.")
    else:
        print(f"Context for '{entity}' is {len(entity_context)} characters long.")
    print()

approach_1_results = normalize(entity_context_pairs)
approach_1_accuracy = summarize_results(approach_1_results)

### Approach 2: Multi-Context Ensemble with Per-Variation Evaluation

**Overview:**  
An ensemble approach that evaluates multiple individual section contexts for each entity by making separate normalization calls for each priority section. This provides a comprehensive view of how each specific section's context influences diagnostic specificity.

**Methodology:**

1. **Individual Section Context Generation:**
   - For each entity in a HIGH-VALUE section, generate separate context variations using the dynamic section priorities
   - Each variation uses ONLY one section's content (not combined)
   - Example for entity in "assessment" section:
     - Call 1: Context = Assessment section only
     - Call 2: Context = HPI section only
     - Call 3: Context = Problem List section only
     - Call 4: Context = History section only
     - Call 5: Context = Physical Exam section only

2. **Dynamic Priority-Based Selection:**
   - Uses Approach 1's research-driven section priorities
   - Different priority orders for entities in different sections
   - Only includes sections that actually exist in the clinical note

3. **Current API Call Pattern (Implementation):**
   - Entity/context variation is currently sent in batches for enrichment API call
   - `ENRICHMENT_BATCH_SIZE = 5`, so 5 entity/context variation is included per enrichment call
   - Example: 10 entities × 5 sections each = ~10 enrichment API calls


4. **Result Depth and Display Behavior:**
   - Enrichment requests use `preferences.size = 3`, so API may return up to 3 matches per entity/section-context pairing
   - The Approach 2 summary view displays only top 2 matches (`top_matches[:2]`) for side-by-side comparison
   - Full returned matches remain available in the raw API response object

**Advantages:**
- ✅ Shows precise impact of each individual section on normalization results
- ✅ Enables evidence-based selection of which sections provide best context
- ✅ Research-validated section priorities ensure relevant sections are evaluated
- ✅ Supports context strategy optimization and refinement

**Limitations:**
- ❌ Higher API costs (5-6× more requests than single-context approaches)
- ❌ Longer processing time due to increased request volume
- ❌ Requires manual review to select final matches

**Key Insight:**  
By evaluating each priority section's context independently, we can identify which specific sections provide the most valuable information for entity normalization. This enables data-driven optimization of context selection strategies.



In [ ]:
def build_context_variations_approach_2(entity: str, section: str, full_text: str, sections: Dict[str, Tuple[int, int]]) -> List[Dict[str, str]]:
    """
    Build individual context variations for a single entity, one per priority section.
    Each variation uses ONLY that section's context, not a combination.
    Returns a list of entity/context pairs for parallel evaluation.
    """
    if section.lower() not in HIGH_VALUE_SECTIONS:
        return []

    normalized_entity_section = normalize_section_name(section)
    sections_by_priority = SECTION_PRIORITIES.get(normalized_entity_section, DEFAULT_PRIORITIES)

    available_sections = {}
    for section_name, section_range in sections.items():
        normalized_name = normalize_section_name(section_name)
        available_sections[normalized_name] = (section_name, section_range)

    context_variations = []
    MAX_CONTEXT_LENGTH = 1000

    for priority_section in sections_by_priority:
        if priority_section in available_sections:
            actual_section_name, (start, end) = available_sections[priority_section]
            section_text = full_text[start:end].strip()
            if len(section_text) > MAX_CONTEXT_LENGTH:
                section_text = section_text[:MAX_CONTEXT_LENGTH]
            context_variations.append({
                "entity": entity,
                "context": section_text,
                "strategy": f"section_{priority_section}",
                "description": f"Context: {actual_section_name}"
            })

    return context_variations

all_entity_context_variations = []
entity_to_variations_map = {}

for idx, (entity, section) in enumerate(entities_df[['entity', 'section']].itertuples(index=False)):
    print(f"\nGenerating context variations for entity '{entity}' in section '{section}'")
    variations = build_context_variations_approach_2(entity, section, clinical_note, sections)

    if variations:
        entity_to_variations_map[entity] = {
            "start_index": len(all_entity_context_variations),
            "count": len(variations),
            "section": section
        }
        all_entity_context_variations.extend(variations)
        print(f"  Generated {len(variations)} context variations")
        for var in variations:
            print(f"    - {var['strategy']}: {var['description']} ({len(var['context'])} chars)")
    else:
        entity_to_variations_map[entity] = {
            "start_index": len(all_entity_context_variations),
            "count": 1,
            "section": section
        }
        all_entity_context_variations.append({
            "entity": entity,
            "context": "",
            "strategy": "no_context",
            "description": "No context (not in high-value section)"
        })
        print("  No variations generated (not in high-value section) - will normalize without context")

print(f"\n{'='*80}")
print(f"TOTAL VARIATIONS GENERATED: {len(all_entity_context_variations)}")
print(f"Entities with variations: {len(entity_to_variations_map)}")
print(f"{'='*80}\n")

if all_entity_context_variations:
    approach_2_results = normalize(all_entity_context_variations)

    # Custom summarization for Approach 2 to show variations side-by-side
    def summarize_approach_2_results(results: Dict, entity_map: Dict):
        """
        Summarize results grouped by entity, showing all context variations side-by-side.
        """
        from IPython.display import display, HTML

        requests = results.get("requests", [])

        found_icd10_codes = []
        for request in requests:
            top_matches = request.get("response", {}).get("items", [])
            for match in top_matches:
                icd10_codes = match.get("metadata", {}).get("mappings", {}).get("icd10cm", {}).get("codes", [])
                for code_obj in icd10_codes:
                    found_icd10_codes.append(code_obj["code"])

        print("\n=== Approach 2: Multi-Context Ensemble Results ===\n")

        for entity, info in entity_map.items():
            start_idx = info["start_index"]
            count = info["count"]
            section = info["section"]

            print(f"\nEntity: '{entity}' (Section: {section})")
            print("-" * 80)

            html = "<style>table.approach2 td {text-align: left; vertical-align: top; white-space: pre-wrap;}</style>"
            html += '<table class="approach2" border="1" style="border-collapse: collapse; width: 100%; margin-bottom: 20px;">'
            html += '<thead><tr>'
            html += '<th style="text-align: left; padding: 8px; background-color: rgba(128, 128, 128, 0.2);">Context Strategy</th>'
            html += '<th style="text-align: left; padding: 8px; background-color: rgba(128, 128, 128, 0.2);">Context Preview</th>'
            html += '<th style="text-align: left; padding: 8px; background-color: rgba(128, 128, 128, 0.2);">Top Matches</th>'
            html += '</tr></thead><tbody>'

            for i in range(count):
                request = requests[start_idx + i]
                strategy = all_entity_context_variations[start_idx + i]["strategy"]
                description = all_entity_context_variations[start_idx + i]["description"]
                context_text = all_entity_context_variations[start_idx + i]["context"]
                context_preview = context_text[:1000] + "..." if len(context_text) > 150 else context_text
                top_matches = request.get("response", {}).get("items", [])

                html += '<tr>'
                html += f'<td style="padding: 8px; border: 1px solid #ddd;"><b>{strategy}</b><br/>{description}</td>'
                html += f'<td style="padding: 8px; border: 1px solid #ddd;">{context_preview}</td>'

                if top_matches:
                    matches_html = ""
                    for rank, match in enumerate(top_matches[:2], 1):
                        default_title = match.get("default_lexical_title", "N/A")
                        icd10_codes = match.get("metadata", {}).get("mappings", {}).get("icd10cm", {}).get("codes", [])
                        icd10_codes_str = ", ".join([code["code"] for code in icd10_codes]) if icd10_codes else "None"
                        score = match.get("score", "N/A")
                        score_str = f"{score:.2f}" if isinstance(score, (int, float)) else score

                        matches_html += f"{rank}. {default_title}\n   ICD-10: {icd10_codes_str}\n   Score: {score_str}"
                        if rank < min(len(top_matches), 2):
                            matches_html += "\n\n"
                    html += f'<td style="padding: 8px; border: 1px solid #ddd;">{matches_html}</td>'
                else:
                    explanation = request.get("response", {}).get("explanation", "No match")
                    html += f'<td style="padding: 8px; border: 1px solid #ddd;">No match found<br/>({explanation})</td>'

                html += '</tr>'

            html += '</tbody></table>'
            display(HTML(html))

        accuracy_results = None
        if 'expected_icd10_codes' in globals() and expected_icd10_codes:
            accuracy_results = calculate_accuracy(expected_icd10_codes, found_icd10_codes)
            print(json.dumps(accuracy_results, indent=2))

        return accuracy_results

    approach_2_accuracy = summarize_approach_2_results(approach_2_results, entity_to_variations_map)
else:
    print("No context variations generated - all entities are in non-high-value sections")
    approach_2_results = {"summary": {"count": 0, "error_count": 0}, "requests": []}
    approach_2_accuracy = {
        "accuracy_percentage": 0.0,
        "true_positives": 0,
        "false_negatives": 0,
        "false_positives": 0,
        "expected_total": 0,
        "found_total": 0
    }

## Next Steps

1. Test with real clinical notes and entity extraction results
2. Measure the impact of different context strategies on normalization accuracy
3. Implement LLM integration if budget allows
4. Create evaluation metrics to compare approaches
5. Optimize context window sizes and pruning strategies

In [ ]:
# Compile available approach results into a single dictionary
all_results = {}

for approach_num in [0, 1, 2]:
    key = f"approach_{approach_num}_accuracy"
    if key in globals():
        all_results[f"{note_id}-approach-{approach_num}"] = globals()[key]

print("\n" + "="*80)
print("SUMMARY: AVAILABLE APPROACHES")
print("="*80)
print(json.dumps(all_results, indent=2))


In [ ]:
# use this code to save results to a JSON file if desired
# with open(f"{note_id}_results.json", 'w', encoding='utf-8') as f:
#     json.dump(all_results, f, ensure_ascii=False, indent=4)

# print(f"Saved to: {note_id}_results.json")

## Summary: Compare All Approaches

Summary of accuracy metrics for all approaches across the current clinical note.


## Export Results to CSV

Export detailed results from each approach to CSV files for further analysis. Each CSV contains the input term, context used, and all matches returned by the normalization API.

In [ ]:
import os
import csv

def export_results_to_csv(results: Dict, approach_name: str, output_folder: str):
    """
    Export normalization results to a CSV file matching the HTML table format.
    One row per input term with all matches combined in a single cell.
    
    Args:
        results: The normalize API response dictionary
        approach_name: Name of the approach (e.g., 'approach-0', 'approach-1')
        output_folder: Directory where CSV files will be saved
    """
    os.makedirs(output_folder, exist_ok=True)

    csv_filename = f"{note_id}_{approach_name}_results.csv"
    csv_path = os.path.join(output_folder, csv_filename)

    requests = results.get("requests", [])

    rows = []
    for request in requests:
        input_term = request.get("input_term", "")
        context_text = request.get("context", {}).get("source_text", "")
        top_matches = request.get("response", {}).get("items", [])

        if top_matches:
            matches_text = ""
            for rank, match in enumerate(top_matches, 1):
                default_title = match.get("default_lexical_title", "N/A")
                icd10_codes = match.get("metadata", {}).get("mappings", {}).get("icd10cm", {}).get("codes", [])
                icd10_codes_str = ", ".join([code["code"] for code in icd10_codes]) if icd10_codes else "None"
                score = match.get("score", "N/A")
                score_str = f"{score:.2f}" if isinstance(score, (int, float)) else score

                matches_text += f"{rank}. {default_title}\n   ICD-10: {icd10_codes_str}\n   Score: {score_str}"
                if rank < len(top_matches):
                    matches_text += "\n\n"
        else:
            explanation = request.get("response", {}).get("explanation", "No match")
            matches_text = f"No match found\n({explanation})"

        rows.append({
            "Input Term": input_term,
            "Context": context_text if context_text else "None",
            "Matches": matches_text
        })

    if rows:
        fieldnames = ["Input Term", "Context", "Matches"]
        with open(csv_path, 'w', encoding='utf-8', newline='') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)

        print(f"✓ Exported {len(rows)} result rows to: {csv_path}")
        return csv_path
    else:
        print(f"⚠ No results to export for {approach_name}")
        return None

output_folder = "results_export"
print(f"Creating output folder: {output_folder}")
print("="*80)

exported_files = []
for approach_num in [0, 1, 2]:
    results_var = f"approach_{approach_num}_results"
    if results_var in globals():
        file_path = export_results_to_csv(globals()[results_var], f"approach-{approach_num}", output_folder)
        if file_path:
            exported_files.append(file_path)

print("="*80)
print(f"\n✓ Export complete! {len(exported_files)} CSV files created in '{output_folder}/' folder")
print("\nExported files:")
for file_path in exported_files:
    print(f"  - {file_path}")
